In [1]:
# ============================================================
# ResNet18 Training Script (Thesis Quality)
# ============================================================

# ▶ Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ============================================================
# 📌 Directories
# ============================================================

train_dir = "/content/drive/MyDrive/processed_data/train"
val_dir   = "/content/drive/MyDrive/processed_data/test"

# ============================================================
# 📌 Data Augmentation + Normalization (To avoid overfitting)
# ============================================================

mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

# ============================================================
# 📌 Load Dataset
# ============================================================

train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset   = datasets.ImageFolder(val_dir, transform=val_transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False)

num_classes = len(train_dataset.classes)
print("Number of Classes:", num_classes)

# ============================================================
# 📌 Load Pretrained ResNet18
# ============================================================

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Add Dropout to avoid overfitting
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.fc.in_features, num_classes)
)

model = model.to(device)

# ============================================================
# 📌 Loss + Optimizer (Weight decay for regularization)
# ============================================================

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

# ============================================================
# 📌 Early Stopping Setup
# ============================================================

best_val_acc = 0.0
patience = 5
no_improve = 0

save_path = "/content/drive/MyDrive/Model/resnet18_model.pth"
os.makedirs("/content/drive/MyDrive/Model", exist_ok=True)

# ============================================================
# 📌 Training Loop
# ============================================================

num_epochs = 50  # thesis level training

for epoch in range(num_epochs):

    model.train()
    running_loss = 0.0
    correct_train = 0

    # -------- TRAINING ----------
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        _, preds = torch.max(outputs, 1)
        correct_train += torch.sum(preds == labels).item()

    train_loss = running_loss / len(train_dataset)
    train_acc = correct_train / len(train_dataset)

    # -------- VALIDATION ----------
    model.eval()
    correct_val = 0

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)

            outputs = model(imgs)
            _, preds = torch.max(outputs, 1)
            correct_val += torch.sum(preds == labels).item()

    val_acc = correct_val / len(val_dataset)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    # -------- EARLY STOPPING ----------
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        no_improve = 0
        torch.save(model.state_dict(), save_path)
        print(f"✓ Model Improved & Saved! Best Val Accuracy: {best_val_acc:.4f}")
    else:
        no_improve += 1
        if no_improve >= patience:
            print("⭕ Early Stopping Triggered!")
            break

print("Training Finished!")
print(f"Best Model Saved at: {save_path}")


Mounted at /content/drive
Using device: cpu
Number of Classes: 2
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 165MB/s]


Epoch [1/50] Train Loss: 0.4139 | Train Acc: 0.8087 | Val Acc: 0.9425
✓ Model Improved & Saved! Best Val Accuracy: 0.9425
Epoch [2/50] Train Loss: 0.2635 | Train Acc: 0.8906 | Val Acc: 0.9500
✓ Model Improved & Saved! Best Val Accuracy: 0.9500
Epoch [3/50] Train Loss: 0.1879 | Train Acc: 0.9225 | Val Acc: 0.9600
✓ Model Improved & Saved! Best Val Accuracy: 0.9600
Epoch [4/50] Train Loss: 0.1760 | Train Acc: 0.9244 | Val Acc: 0.9625
✓ Model Improved & Saved! Best Val Accuracy: 0.9625
Epoch [5/50] Train Loss: 0.1379 | Train Acc: 0.9513 | Val Acc: 0.9700
✓ Model Improved & Saved! Best Val Accuracy: 0.9700
Epoch [6/50] Train Loss: 0.1391 | Train Acc: 0.9494 | Val Acc: 0.9675
Epoch [7/50] Train Loss: 0.1411 | Train Acc: 0.9431 | Val Acc: 0.9525
Epoch [8/50] Train Loss: 0.1136 | Train Acc: 0.9556 | Val Acc: 0.9825
✓ Model Improved & Saved! Best Val Accuracy: 0.9825
Epoch [9/50] Train Loss: 0.1092 | Train Acc: 0.9563 | Val Acc: 0.9700
Epoch [10/50] Train Loss: 0.1310 | Train Acc: 0.9450 | Val